# 传感器地图可视化

可视化 sensors_common_2025.csv 中的传感器位置

In [3]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster
import os

OUTPUT_DIR = "../output/maps"
META_DIR = "../d03_meta_processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [4]:
# 加载数据
df = pd.read_csv(os.path.join(META_DIR, 'sensors_common_2025.csv'), dtype={'ID': str, 'Fwy': str})
print(f"传感器数: {len(df)}")
df.head()

传感器数: 1830


,ID,Fwy,Dir,District,County,City,State_PM,Abs_PM,Latitude,Longitude,Length,Type,Lanes,Name,User_ID_1,User_ID_2,User_ID_3,User_ID_4
0,308511,50,E,3,17,NaN,31.627,60.162,38.761062,-120.569835,3.134,ML,2,Sly Park Rd,1,NaN,NaN,NaN
1,308512,50,W,3,17,NaN,31.627,60.166,38.761182,-120.569866,5.000,ML,2,Sly Park Rd,1,NaN,NaN,NaN
2,311831,5,S,3,67,NaN,10.896,506.189,38.409782,-121.484120,NaN,OR,1,Elk Grove Blvd to 5SB Loop,1,NaN,NaN,NaN
3,311832,5,S,3,67,NaN,10.896,506.189,38.409782,-121.484120,NaN,FR,1,5SB to Elk Grove Blvd,1,NaN,NaN,NaN
4,311844,5,N,3,67,NaN,11.08,506.373,38.412421,-121.484289,NaN,OR,2,Elk Grove Blvd 5NB Slip,1,NaN,NaN,NaN


In [5]:
def create_sensor_map(df, use_cluster=True):
    """
    创建传感器地图
    """
    # 过滤有效坐标
    df_valid = df.dropna(subset=['Latitude', 'Longitude'])
    df_valid = df_valid[(df_valid['Latitude'] != 0) & (df_valid['Longitude'] != 0)]
    print(f"有效坐标: {len(df_valid)} / {len(df)}")
    
    # 中心点
    center_lat = df_valid['Latitude'].mean()
    center_lon = df_valid['Longitude'].mean()
    
    # 创建地图
    m = folium.Map(location=[center_lat, center_lon], zoom_start=9, tiles='OpenStreetMap')
    
    # 类型颜色
    type_colors = {
        'ML': 'blue', 'OR': 'green', 'FR': 'red',
        'HV': 'purple', 'FF': 'orange', 'CD': 'darkblue', 'CH': 'gray'
    }
    
    # 聚类或直接添加
    if use_cluster:
        marker_cluster = MarkerCluster(name='Stations').add_to(m)
        target = marker_cluster
    else:
        target = m
    
    # 添加标记
    for _, row in df_valid.iterrows():
        color = type_colors.get(row['Type'], 'gray')
        
        # 标签内容
        popup_html = f"""
        <b>ID:</b> {row['ID']}<br>
        <b>Fwy:</b> I-{row['Fwy']} {row['Dir']}<br>
        <b>Abs_PM:</b> {row['Abs_PM']}<br>
        <b>Length:</b> {row['Length'] if pd.notna(row['Length']) else 'N/A'}<br>
        <b>Type:</b> {row['Type']}<br>
        <b>Lanes:</b> {int(row['Lanes']) if pd.notna(row['Lanes']) else 'N/A'}<br>
        <b>Name:</b> {row['Name'] if pd.notna(row['Name']) else 'N/A'}
        """
        
        folium.CircleMarker(
            location=[row['Latitude'], row['Longitude']],
            radius=6,
            popup=folium.Popup(popup_html, max_width=300),
            tooltip=f"I-{row['Fwy']}{row['Dir']} | {row['ID']} | PM:{row['Abs_PM']:.1f}",
            color=color,
            fill=True,
            fillColor=color,
            fillOpacity=0.7
        ).add_to(target)
    
    # 图例
    legend_html = '''
    <div style="position: fixed; bottom: 50px; left: 50px; z-index: 1000;
                background-color: white; padding: 10px; border-radius: 5px;
                border: 2px solid gray; font-size: 12px;">
    <b>Type</b><br>
    <i style="background:blue; width:12px; height:12px; display:inline-block;"></i> ML<br>
    <i style="background:green; width:12px; height:12px; display:inline-block;"></i> OR<br>
    <i style="background:red; width:12px; height:12px; display:inline-block;"></i> FR<br>
    <i style="background:purple; width:12px; height:12px; display:inline-block;"></i> HV<br>
    <i style="background:orange; width:12px; height:12px; display:inline-block;"></i> FF
    </div>
    '''
    m.get_root().html.add_child(folium.Element(legend_html))
    
    return m

In [6]:
# 创建地图（使用聚类）
m = create_sensor_map(df, use_cluster=True)
m

有效坐标: 1828 / 1830


In [7]:
# 保存地图
m.save(os.path.join(OUTPUT_DIR, 'sensors_map.html'))
print("地图已保存: sensors_map.html")

地图已保存: sensors_map.html


In [8]:
# 只看 ML 传感器（不聚类，方便查看详情）
ml_df = df[df['Type'] == 'ML']
print(f"ML 传感器数: {len(ml_df)}")

m_ml = create_sensor_map(ml_df, use_cluster=False)
m_ml

ML 传感器数: 854
有效坐标: 853 / 854
